In [1]:
import sys
import os
sys.path.append('../TCT/')
import TCT
from TCT import translator_kpinfo
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_query
from TCT import TCT_pathfinder
import time
import json

In [2]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources()

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

    # generate a dictionary of API and its predicates
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
Skipping server without x-maturity: {'url': '/sipr'}


In [3]:
# select a list of APIs to use and a list of predicates to use
selected_APIlist = []

if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}

selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)
print(selected_metaKG.shape)

(22568, 5)


In [54]:
subject_name = 'Escherichia coli'
subject_node = 'NCBITaxon:562'
subject_category = ['biolink:OrganismTaxon']
name_resolver.lookup(subject_name, return_top_response=False, return_synonyms=True)

[TranslatorNode(curie='DRUGBANK:DB16539', label='Escherichia coli', types=['biolink:ChemicalEntity', 'biolink:PhysicalEssence', 'biolink:ChemicalOrDrugOrTreatment', 'biolink:ChemicalEntityOrGeneOrGeneProduct', 'biolink:ChemicalEntityOrProteinOrPolypeptide', 'biolink:NamedThing', 'biolink:Entity', 'biolink:PhysicalEssenceOrOccurrent'], synonyms=['Escherichia coli'], curie_synonyms=None, attributes=None, taxa=[]),
 TranslatorNode(curie='MONDO:0020920', label='escherichia coli infection', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=['E COLI INFECT', 'INFECT E COLI', 'colibacillosis', 'Colibacillosis', 'e coli infection', 'coli e infection', 'Escherichia coli', 'E Coli Infection', 'E coli infection', 'E COLI INFECTION', 'E coli Infection', 'Infection;E coli', 'E. coli infection', 'coli e. infection', 'Infection, E coli', 'E. coli Infection', 'infections e coli', 'coli

In [55]:
object_name = 'colorectal cancer'
object_node = 'MONDO:0005575'
object_category = ['biolink:Disease']
name_resolver.lookup(object_name, return_top_response=False, return_synonyms=True)


[TranslatorNode(curie='MONDO:0005575', label='colorectal cancer', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=['CRC', 'colon cancer', 'colorectum cancer', 'colorectal cancer', 'large bowel cancer', 'cancer of colorectum', 'cancer of large colon', 'Cancer of large bowel', 'of large bowel cancer', 'colon cancer, somatic', 'cancer of large bowel', 'large intestine cancer', 'cancer intestine large', 'cancer intestines large', 'cancer of the large bowel', 'cancer of large intestine', 'Cancer of large intestine', 'Malignant tumor large int', 'Malignant Colorectal Tumor', 'Malignant tumour large int', 'colorectal cancer, somatic', 'CA - Cancer of large bowel', 'malignant colorectal tumor', 'Neoplasm malig;bowel;large', 'Malignant Large Bowel Tumor', 'malignant colorectal tumour', 'malignant large bowel tumor', 'malignant large bowel tumour', 'Malignant Colorectal Neoplas

In [47]:
intermediate_categories = ['biolink:Gene', 'biolink:Protein']


In [56]:
# TCT pathfinder pipeline
start_time = time.time()
result1, result2, TCT_path_finder_result = TCT_pathfinder.pathfinder(input_node1_id=subject_node, input_node2_id= object_node, #COVID-19
                                                                            intermediate_categories=intermediate_categories, 
                                                                            APInames=select_APIs, 
                                                                            metaKG=selected_metaKG, 
                                                                            API_predicates=API_predicates, 
                                                                            scoring_method='infores')


end_time = time.time()

TCT_execution_time = end_time - start_time



MONDO:0005575
Microbiome KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!
Automat-robokop(Trapi v1.5.0): Success!
Automat-ubergraph(Trapi v1.5.0): Success!
Clinical Trials KP - TRAPI 1.5.0: Success!
RTX KG2 - TRAPI 1.5.0: Success!


In [57]:
# return results path_finder_result to a json file
import json
with open(f'TCT_path_finder_result__{subject_name.replace(":", "_")}__{object_name.replace(":", "_")}.json', 'w') as f:
    json.dump(TCT_path_finder_result, f, indent=4)
Number_of_paths_TCT = len(TCT_path_finder_result['auxiliary_graphs'])
print(f"Number of paths TCT: {Number_of_paths_TCT}")

Number of paths TCT: 139


In [58]:
# run aragorn pathfinder with the same input and write response to a json file
start_time = time.time()
aragorn_response = TCT_pathfinder.query_aragorn_pathfinder(subject_node, 
                                                           subject_category, 
                                                           object_node, 
                                                           object_category)
# write response to a json file
import json
if 'message' in aragorn_response.json():
    with open('aragorn_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'.json', 'w') as f:
        json.dump(aragorn_response.json()['message'], f, indent=4)
end_time = time.time()
aragorn_execution_time = end_time - start_time
Number_of_paths_aragorn = len(aragorn_response.json()['message']['auxiliary_graphs'])
print(f"Aragorn execution time: {aragorn_execution_time} seconds")
print(f"Number of paths Aragorn: {Number_of_paths_aragorn}")

Aragorn execution time: 45.81710410118103 seconds
Number of paths Aragorn: 1000


In [59]:
start_time = time.time()
arax_response = TCT_pathfinder.query_arax_pathfinder(subject_node, 'biolink:Drug', object_node, 'biolink:Disease')
# write response to a json file
import json
with open('arax_pathfinder_response_'+subject_name.replace(":", "_")+'_'+object_name.replace(":", "_")+'.json', 'w') as f:
    json.dump(arax_response.json()['message'], f, indent=4)
end_time = time.time()
arax_execution_time = end_time - start_time
if arax_response.json()['message'].get('auxiliary_graphs') is not None:
    Number_of_paths_ARAX = len(arax_response.json()['message']['auxiliary_graphs'])
else:
    Number_of_paths_ARAX = 0
print(f"ARAX execution time: {arax_execution_time} seconds")
print(f"Number of paths ARAX: {Number_of_paths_ARAX}")

ARAX execution time: 152.14187097549438 seconds
Number of paths ARAX: 500
